In [4]:
!uv add -q polars numpy SPARQLWrapper gensim python-dotenv kagglehub[polars-datasets] ipywidgets sentence-transformers

__IMPORTING REQUIRED LIBRARIES__

_Required libraries (fix order):_
1. polars (replacing pandas (novelty))
2. numpy
3. gensim
4. sentence-transformers
5. matplotlib
6. scikit-learn
7. os
8. dotenv
9. kagglehub
10. torch (for cuda)

In [3]:
import os
from dotenv import load_dotenv

import polars as pl
import numpy
import gensim
import torch

In [4]:
wv = gensim.models.KeyedVectors.load_word2vec_format('word2vec_format/GoogleNews-vectors-negative300.bin', binary=True)

__LOADING DATASET__

_Dataset Details:_
1. Name - ISOT Fake News Dataset
2. Link - https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets/data
3. Number of rows - 20,826 (True) and 17,903 (False)

In [5]:
load_dotenv()

kaggle_key = os.getenv('KAGGLE_API_TOKEN')
kaggle_username = os.getenv('KAGGLE_USERNAME')

In [6]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [7]:
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets", output_dir='./dataset')

In [8]:
true_df = pl.read_csv('dataset/News _dataset/True.csv')
fake_df = pl.read_csv('dataset/News _dataset/Fake.csv')

In [9]:
true_df.head()

title,text,subject,date
str,str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews""","""December 31, 2017 """
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews""","""December 29, 2017 """
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews""","""December 31, 2017 """
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews""","""December 30, 2017 """
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews""","""December 29, 2017 """


In [10]:
true_df = true_df.drop('date')
true_df.head()

title,text,subject
str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews"""
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews"""
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews"""
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews"""
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews"""


In [11]:
fake_df.head()

title,text,subject,date
str,str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News""","""December 31, 2017"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News""","""December 31, 2017"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News""","""December 30, 2017"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News""","""December 29, 2017"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News""","""December 25, 2017"""


In [12]:
fake_df = fake_df.drop("date")
fake_df.head()

title,text,subject
str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News"""


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
from sentence_transformers import SentenceTransformer

In [10]:
# model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", device=device, trust_remote_code=True)

configuration.py: 0.00B [00:00, ?B/s]

D:\University\Courses\SEM 6\NLP\Project\fake_news_verifier\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sasmi\.cache\huggingface\hub\models--Alibaba-NLP--new-impl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] A new version of the following files was downloaded from https://hug

modeling.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
fake_title_vectors = model.encode(fake_df['title'].to_list(),show_progress_bar=True)
fake_text_vectors = model.encode(fake_df['text'].to_list(), show_progress_bar=True)

In [ ]:
fake_df = fake_df.with_columns(
    title_vectorised = pl.Series(fake_title_vectors),
    text_vectorized = pl.Series(fake_text_vectors),
)

In [ ]:
fake_df.head()